# GNN Feature Extractor
## Heterogeneous Graph Neural Network for Fraud Detection

**Purpose:** Build 64-dim embeddings per claim using a heterogeneous graph.
Embeddings capture fraud-ring signals invisible to row-level models.

**Output:** `gnn_embeddings.parquet` — one row per Claim_No with 64 embedding columns.
Load this in `scale pos fraud.ipynb` and concatenate to existing features.

**Graph Schema:**
- Nodes: Claim · Hospital (HID_anon) · Doctor (Treating_Dr) · Patient (CID_anon) · Disease (Disease_Category) · Pincode · Employer (Policy_number)
- Edges: Claim->Hospital · Claim->Doctor · Claim->Patient · Claim->Disease · Claim->Pincode · Claim->Employer
- Model: 3-layer HeteroSAGE (inductive — handles new test nodes)
- Hardware: RTX 4070 12GB (~3-4GB VRAM usage)

**Key signals captured:**
- Diagnosis Herfindahl index per hospital (#1 fraud ring signal per research)
- Doctor exclusivity score (multi-hospital = flag)
- Expected LOS per disease (upcoding signal, primary OIG audit flag)
- Employer x hospital flocking (group insurance fraud ring)
- TTD + Hospital_Cash_Allowance (most fraud-prone add-ons per RGA 2024)

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, to_hetero
from torch_geometric.transforms import ToUndirected
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import average_precision_score, roc_auc_score
import warnings, time, os
warnings.filterwarnings("ignore")

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyG version check OK")

c:\Users\mitul\.micromamba\envs\set\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4070
VRAM: 12.9 GB
PyG version check OK


In [2]:
# Load data and apply time split
print("Loading data...")
df = pd.read_csv(r"C:\\Users\\mitul\\OneDrive\\Documents\\arya\\fraud\\CFM_anon_final.csv", low_memory=False)
print(f"Loaded {len(df):,} rows")

df["data_created_at_parsed"] = pd.to_datetime(df["data_created_at"], format="ISO8601")

# Time split — same as main pipeline
train_mask = df["data_created_at_parsed"] < "2025-07-01"
test_mask  = df["data_created_at_parsed"] >= "2025-07-01"
print(f"Train: {train_mask.sum():,} | Test: {test_mask.sum():,}")

# Remove leaky columns
leaky = ["Investigation Outcome","Investigation Outcome1","Old_Target",
         "Final Status","Investigated","Assign Date","Claim Type","Approved_Amount_INR"]
df = df.drop(columns=[c for c in leaky if c in df.columns])

# Targets (use train labels only for GNN training)
y_inv   = df["Target_as_investigation"].fillna(0).astype(int).values
y_fraud = df["Target_as_fraud"].fillna(0).astype(int).values

print(f"Investigation positives (train): {y_inv[train_mask].sum():,}")
print(f"Fraud positives (train): {y_fraud[train_mask].sum():,}")

# Keep Claim_No as index for final output alignment
claim_ids = df["Claim_No"].values.copy()
N = len(df)
print(f"Total claims (nodes): {N:,}")

Loading data...
Loaded 474,619 rows
Train: 415,751 | Test: 58,868
Investigation positives (train): 27,930
Fraud positives (train): 11,693
Total claims (nodes): 474,619


In [ ]:
# Build integer index maps for each entity type
# NOTE: CID_anon = Corporate/Client ID (33 insurance clients) — NOT patient ID.
#       The dataset has no individual member/patient identifier.
#       We keep CID_anon as a "client" node (captures insurer-client-level fraud rates).
print("Building entity index maps...")

def make_index(series, fill_missing="UNKNOWN"):
    vals = series.fillna(fill_missing).astype(str)
    le = LabelEncoder()
    le.fit(vals)
    return le.transform(vals), len(le.classes_)

# --- Node types ---
claim_idx  = np.arange(N)   # claims are already 0..N-1

hosp_idx,     n_hosp     = make_index(df["HID_anon"])
doctor_idx,   n_doctor   = make_index(df["Treating_Dr"])
client_idx,   n_client   = make_index(df["CID_anon"])       # 33 corporate insurance clients
pincode_idx,  n_pincode  = make_index(df["Pincode"])
employer_idx, n_employer = make_index(df["Policy_number"])  # 48K employer group policies

# Disease: use Disease_Category; fall back to Final_Diagnosis first char if missing
disease_col = df["Disease_Category"].fillna(df["Final_Diagnosis"].str[:3]).fillna("UNK")
disease_idx, n_disease = make_index(disease_col)

print(f"Node counts:")
print(f"  Claims   : {N:,}")
print(f"  Hospitals: {n_hosp:,}")
print(f"  Doctors  : {n_doctor:,}")
print(f"  Clients  : {n_client:,}   (CID_anon = corporate insurance clients)")
print(f"  Pincodes : {n_pincode:,}")
print(f"  Diseases : {n_disease:,}")
print(f"  Employers: {n_employer:,}")
print(f"  Total    : {N + n_hosp + n_doctor + n_client + n_pincode + n_disease + n_employer:,}")

entity_idx = {
    "claim":    claim_idx,
    "hosp":     hosp_idx,
    "doctor":   doctor_idx,
    "client":   client_idx,
    "pincode":  pincode_idx,
    "disease":  disease_idx,
    "employer": employer_idx,
}
entity_counts = {
    "claim": N, "hosp": n_hosp, "doctor": n_doctor,
    "client": n_client, "pincode": n_pincode, "disease": n_disease,
    "employer": n_employer,
}
print("Entity maps built")

In [ ]:
# Build node feature tensors for each node type
# Rule: aggregations computed on TRAIN ONLY to prevent leakage
# All node feature arrays are RobustScaled — critical for GNN gradient stability
print("Building node features (train-only aggregations)...")

# ── Pre-compute derived columns on full df ────────────────────────────────────
df["hosp_days"] = (
    pd.to_datetime(df["Actual_Date_of_Discharge"], errors="coerce") -
    pd.to_datetime(df["Actual_Date_of_Admission"],  errors="coerce")
).dt.days.clip(0, 90).fillna(0)

df["days_since_inception"] = (
    pd.to_datetime(df["Actual_Date_of_Admission"], errors="coerce") -
    pd.to_datetime(df["Risk_Inception_Date"],      errors="coerce")
).dt.days.clip(0, 3650).fillna(0)

# Add-on cover columns — most fraud-prone per RGA 2024 survey
addon_cols = [c for c in ["TTD","PTD","PPD","Death","Accidental_Medical_Expenses",
                           "Accidental_Hospitalisation","Hospital_Cash_Allowance",
                           "Broken_Bones","Road_Ambulance_Cover",
                           "Child_Education_Support","Life_Support_Benefit"] if c in df.columns]
df["num_addon_covers_claimed"] = (df[addon_cols].fillna(0) > 0).sum(axis=1).astype(float)
df["addon_total"]              = df[addon_cols].fillna(0).sum(axis=1)
df["addon_to_base_ratio"]      = (
    df["addon_total"] / df["Claimed_Amt"].replace(0, np.nan)
).fillna(0).clip(0, 10)

# Categorical columns → label-encoded integers
for cat_col in ["Gender_of_Patient","Relationship_of_Patient_with_Employee",
                "Type_of_Hospital","Room_Category","Nature_of_Treatment",
                "Treatment_Type","KYC_Flag","Policy_Type___Floater_or_Standard"]:
    if cat_col in df.columns:
        le_c = LabelEncoder()
        vals = df[cat_col].fillna("UNKNOWN").astype(str)
        le_c.fit(vals)
        df[f"enc_{cat_col}"] = le_c.transform(vals).astype(float)

train_df = df[train_mask].copy()

def scale_arr(raw, n_nodes):
    """RobustScale a node feature array."""
    arr = np.zeros((n_nodes, raw.shape[1]), dtype=np.float32)
    if len(raw) == 0:
        return arr
    scl = RobustScaler()
    scaled = scl.fit_transform(raw.astype(np.float32))
    np.clip(scaled, -10, 10, out=scaled)
    return scaled

# ── Claim node features ───────────────────────────────────────────────────────
base_num_cols = [
    "Age_of_Patient_Years","Claimed_Amt","Total_Bill","Sum_Insured",
    "Balance_Sum_Insured","Claim_Reserve","Buffer_Available","Buffer_Consumed",
    "Balance_Buffer","Actual_Room_Charges","Doctor","Anaesthetist","Medicines",
    "Surgeon_Fees","Radiology","Pathalogy","OT","Miscellaneous",
    "Ambulance","Cardiology","Equipment","Package_Amount",
    "Hospicash_Sum_insured","Accidental_Hospitalization_suminsured",
    "Critical_Illness_Utilization_Amount",
    "No_of_members_present_in_Group_corporate_policy","Co_payment",
    "TTD","PTD","PPD","Death","Accidental_Hospitalisation","Hospital_Cash_Allowance",
    "Accidental_Medical_Expenses","Broken_Bones",
    "hosp_days","days_since_inception",
    "num_addon_covers_claimed","addon_total","addon_to_base_ratio",
]
enc_cols  = [c for c in df.columns if c.startswith("enc_")]
num_cols  = [c for c in (base_num_cols + enc_cols) if c in df.columns]

claim_feat = df[num_cols].copy()
train_meds = train_df[num_cols].median()
claim_feat = claim_feat.fillna(train_meds).replace([float("inf"), float("-inf")], 0)
claim_scaler   = RobustScaler()
claim_feat_arr = claim_scaler.fit_transform(claim_feat.values.astype(np.float32))
np.clip(claim_feat_arr, -10, 10, out=claim_feat_arr)

# ── Helpers ───────────────────────────────────────────────────────────────────
def herfindahl(series):
    if len(series) == 0: return 0.0
    counts = series.value_counts(normalize=True)
    return float((counts ** 2).sum()) if len(counts) else 0.0

def top_pct(series):
    if len(series) == 0: return 0.0
    vc = series.value_counts()
    return float(vc.iloc[0] / len(series)) if len(vc) else 0.0

# ── Hospital node features ────────────────────────────────────────────────────
train_df_h    = train_df.assign(h=hosp_idx[train_mask])
hosp_diag_hhi = train_df_h.groupby("h")["Disease_Category"].apply(herfindahl).rename("hosp_diag_hhi")
hosp_agg = (train_df_h.groupby("h").agg(
                hosp_fraud_rate    =("Target_as_fraud",        "mean"),
                hosp_inv_rate      =("Target_as_investigation", "mean"),
                hosp_claim_mean    =("Claimed_Amt",             "mean"),
                hosp_claim_std     =("Claimed_Amt",             "std"),
                hosp_claim_count   =("Claimed_Amt",             "count"),
                hosp_uniq_doctors  =("Treating_Dr",             "nunique"),
                hosp_uniq_diagnoses=("Disease_Category",        "nunique"),
                hosp_uniq_cities   =("Insured_City",            "nunique"),
                hosp_mean_LOS      =("hosp_days",               "mean"),
            ).fillna(0))
hosp_agg = hosp_agg.join(hosp_diag_hhi, how="left").fillna(0)
hosp_feat_arr = np.zeros((n_hosp, hosp_agg.shape[1]), dtype=np.float32)
hosp_feat_arr[hosp_agg.index] = hosp_agg.values.astype(np.float32)
hosp_feat_arr = scale_arr(hosp_feat_arr, n_hosp)

# ── Doctor node features ──────────────────────────────────────────────────────
train_df_d   = train_df.assign(d=doctor_idx[train_mask])
doc_excl     = train_df_d.groupby("d")["HID_anon"].apply(top_pct).rename("doc_exclusivity")
doc_diag_hhi = train_df_d.groupby("d")["Disease_Category"].apply(herfindahl).rename("doc_diag_hhi")
doc_agg = (train_df_d.groupby("d").agg(
               doc_fraud_rate  =("Target_as_fraud",        "mean"),
               doc_inv_rate    =("Target_as_investigation", "mean"),
               doc_claim_mean  =("Claimed_Amt",             "mean"),
               doc_claim_count =("Claimed_Amt",             "count"),
               doc_hosp_count  =("HID_anon",                "nunique"),
               doc_diag_count  =("Disease_Category",        "nunique"),
           ).fillna(0))
doc_agg = doc_agg.join(doc_excl, how="left").join(doc_diag_hhi, how="left").fillna(0)
doc_feat_arr = np.zeros((n_doctor, doc_agg.shape[1]), dtype=np.float32)
doc_feat_arr[doc_agg.index] = doc_agg.values.astype(np.float32)
doc_feat_arr = scale_arr(doc_feat_arr, n_doctor)

# ── Client node features (CID_anon = 33 corporate insurance clients) ──────────
# Each client node aggregates all employer policies under that corporate account.
# Captures client-level fraud rings (one corporate client systematically defrauded).
train_df_c    = train_df.assign(c=client_idx[train_mask])
cli_top_hosp  = train_df_c.groupby("c")["HID_anon"].apply(top_pct).rename("cli_top_hosp_pct")
cli_agg = (train_df_c.groupby("c").agg(
               cli_fraud_rate      =("Target_as_fraud",        "mean"),
               cli_inv_rate        =("Target_as_investigation", "mean"),
               cli_claim_count     =("Claimed_Amt",             "count"),
               cli_claim_mean      =("Claimed_Amt",             "mean"),
               cli_uniq_hospitals  =("HID_anon",                "nunique"),
               cli_uniq_employers  =("Policy_number",           "nunique"),
               cli_uniq_diseases   =("Disease_Category",        "nunique"),
               cli_mean_LOS        =("hosp_days",               "mean"),
           ).fillna(0))
cli_agg = cli_agg.join(cli_top_hosp, how="left").fillna(0)
client_feat_arr = np.zeros((n_client, cli_agg.shape[1]), dtype=np.float32)
client_feat_arr[cli_agg.index] = cli_agg.values.astype(np.float32)
client_feat_arr = scale_arr(client_feat_arr, n_client)

# ── Pincode node features ─────────────────────────────────────────────────────
pin_agg = (train_df.assign(p=pincode_idx[train_mask]).groupby("p").agg(
               pin_fraud_rate  =("Target_as_fraud",  "mean"),
               pin_claim_count =("Claimed_Amt",       "count"),
               pin_hosp_count  =("HID_anon",          "nunique"),
           ).fillna(0))
pin_feat_arr = np.zeros((n_pincode, pin_agg.shape[1]), dtype=np.float32)
pin_feat_arr[pin_agg.index] = pin_agg.values.astype(np.float32)
pin_feat_arr = scale_arr(pin_feat_arr, n_pincode)

# ── Disease node features ─────────────────────────────────────────────────────
dis_agg = (train_df.assign(d=disease_idx[train_mask]).groupby("d").agg(
               dis_fraud_rate   =("Target_as_fraud",  "mean"),
               dis_claim_mean   =("Claimed_Amt",       "mean"),
               dis_claim_count  =("Claimed_Amt",       "count"),
               dis_expected_LOS =("hosp_days",         "median"),
               dis_LOS_std      =("hosp_days",         "std"),
           ).fillna(0))
dis_feat_arr = np.zeros((n_disease, dis_agg.shape[1]), dtype=np.float32)
dis_feat_arr[dis_agg.index] = dis_agg.values.astype(np.float32)
dis_feat_arr = scale_arr(dis_feat_arr, n_disease)

# ── Employer node features ────────────────────────────────────────────────────
train_df_e   = train_df.assign(e=employer_idx[train_mask])
emp_flocking = train_df_e.groupby("e")["HID_anon"].apply(top_pct).rename("emp_top_hosp_pct")
emp_agg = (train_df_e.groupby("e").agg(
               emp_fraud_rate    =("Target_as_fraud",        "mean"),
               emp_inv_rate      =("Target_as_investigation", "mean"),
               emp_claim_count   =("Claimed_Amt",             "count"),
               emp_uniq_hospitals=("HID_anon",                "nunique"),
               emp_claim_mean    =("Claimed_Amt",             "mean"),
           ).fillna(0))
emp_agg = emp_agg.join(emp_flocking, how="left").fillna(0)
emp_feat_arr = np.zeros((n_employer, emp_agg.shape[1]), dtype=np.float32)
emp_feat_arr[emp_agg.index] = emp_agg.values.astype(np.float32)
emp_feat_arr = scale_arr(emp_feat_arr, n_employer)

print(f"Claim   feat: {claim_feat_arr.shape}")
print(f"Hosp    feat: {hosp_feat_arr.shape}")
print(f"Doctor  feat: {doc_feat_arr.shape}")
print(f"Client  feat: {client_feat_arr.shape}   (33 corporate clients)")
print(f"Pincode feat: {pin_feat_arr.shape}")
print(f"Disease feat: {dis_feat_arr.shape}")
print(f"Employer feat:{emp_feat_arr.shape}")
print("Node features ready")

In [ ]:
# Build PyG HeteroData object
print("Building heterogeneous graph...")

data = HeteroData()

# Node features
data["claim"].x    = torch.tensor(claim_feat_arr,   dtype=torch.float)
data["hosp"].x     = torch.tensor(hosp_feat_arr,    dtype=torch.float)
data["doctor"].x   = torch.tensor(doc_feat_arr,     dtype=torch.float)
data["client"].x   = torch.tensor(client_feat_arr,  dtype=torch.float)  # 33 corporate clients
data["pincode"].x  = torch.tensor(pin_feat_arr,     dtype=torch.float)
data["disease"].x  = torch.tensor(dis_feat_arr,     dtype=torch.float)
data["employer"].x = torch.tensor(emp_feat_arr,     dtype=torch.float)

# Labels on claim nodes (for GNN training)
data["claim"].y_fraud = torch.tensor(y_fraud, dtype=torch.long)
data["claim"].y_inv   = torch.tensor(y_inv,   dtype=torch.long)

# Train / test masks on claim nodes
data["claim"].train_mask = torch.tensor(train_mask.values, dtype=torch.bool)
data["claim"].test_mask  = torch.tensor(test_mask.values,  dtype=torch.bool)

# Edges (directed: claim -> entity)
ci = torch.tensor(claim_idx, dtype=torch.long)
data["claim", "at",        "hosp"    ].edge_index = torch.stack([ci, torch.tensor(hosp_idx,     dtype=torch.long)])
data["claim", "by",        "doctor"  ].edge_index = torch.stack([ci, torch.tensor(doctor_idx,   dtype=torch.long)])
data["claim", "client_of", "client"  ].edge_index = torch.stack([ci, torch.tensor(client_idx,   dtype=torch.long)])
data["claim", "from",      "pincode" ].edge_index = torch.stack([ci, torch.tensor(pincode_idx,  dtype=torch.long)])
data["claim", "disease",   "disease" ].edge_index = torch.stack([ci, torch.tensor(disease_idx,  dtype=torch.long)])
data["claim", "under",     "employer"].edge_index = torch.stack([ci, torch.tensor(employer_idx, dtype=torch.long)])

# Make undirected so messages flow both ways (entity -> claim too)
data = ToUndirected()(data)

print(data)
print(f"\nEdge types: {len(data.edge_types)}")

# Memory estimate
total_nodes = sum(data[nt].x.shape[0] for nt in data.node_types)
total_edges = sum(data[et].edge_index.shape[1] for et in data.edge_types)
print(f"\nTotal nodes: {total_nodes:,} | Total edges: {total_edges:,}")
print(f"Est. GPU memory: ~{(total_nodes * 64 * 4 + total_edges * 8) / 1e9:.2f} GB")
print("Graph built")

In [ ]:
# Define ResHeteroSAGE — 4-layer with residual connections, LayerNorm, Dropout
# Architecture: Input -> 256 -> 256(res) -> 256(res) -> 128(out)
# Improvements over v1:
#   - 4 layers vs 3 (deeper = richer neighborhood aggregation)
#   - 256 hidden vs 128 (2x capacity)
#   - Residual connections on layers 2-3 (prevents vanishing gradients)
#   - LayerNorm after each layer (stabilises heterogeneous feature scales)
#   - ELU activation (smoother gradients for sparse graphs)
#   - Dropout 0.3 (regularisation — needed with many entity types)
#   - 128-dim output vs 64 (richer embeddings for downstream models)
from torch_geometric.nn import SAGEConv, Linear, HeteroConv

class ResHeteroSAGE(torch.nn.Module):
    def __init__(self, hidden_dim=256, out_dim=128, metadata=None, dropout=0.3):
        super().__init__()
        node_types, edge_types = metadata

        # Layer 1: lazy input (-1) -> hidden (different input dims per node type)
        self.conv1 = HeteroConv(
            {et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="mean"
        )
        # Layer 2: hidden -> hidden  (residual)
        self.conv2 = HeteroConv(
            {et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="mean"
        )
        # Layer 3: hidden -> hidden  (residual)
        self.conv3 = HeteroConv(
            {et: SAGEConv((-1, -1), hidden_dim) for et in edge_types}, aggr="mean"
        )
        # Layer 4: hidden -> out (no residual — dimension changes)
        self.conv4 = HeteroConv(
            {et: SAGEConv((-1, -1), out_dim) for et in edge_types}, aggr="mean"
        )

        # Per-node-type LayerNorm (after each hidden layer)
        self.norm1 = torch.nn.ModuleDict({nt: torch.nn.LayerNorm(hidden_dim) for nt in node_types})
        self.norm2 = torch.nn.ModuleDict({nt: torch.nn.LayerNorm(hidden_dim) for nt in node_types})
        self.norm3 = torch.nn.ModuleDict({nt: torch.nn.LayerNorm(hidden_dim) for nt in node_types})

        self.drop = torch.nn.Dropout(p=dropout)

        # Classification heads
        self.head_fraud = Linear(out_dim, 2)
        self.head_inv   = Linear(out_dim, 2)

    def forward(self, x_dict, edge_index_dict):
        # --- Layer 1 (no residual: input dims differ per node type) ---
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: self.drop(F.elu(self.norm1[k](v))) for k, v in h.items()}

        # --- Layer 2 (residual) ---
        h2 = self.conv2(h, edge_index_dict)
        h2 = {k: self.drop(F.elu(self.norm2[k](v))) for k, v in h2.items()}
        h = {k: h[k] + h2[k] for k in h}   # residual add

        # --- Layer 3 (residual) ---
        h3 = self.conv3(h, edge_index_dict)
        h3 = {k: self.drop(F.elu(self.norm3[k](v))) for k, v in h3.items()}
        h = {k: h[k] + h3[k] for k in h}   # residual add

        # --- Layer 4 (projection to output dim) ---
        h = self.conv4(h, edge_index_dict)
        return h

    def classify_fraud(self, embeddings):
        return self.head_fraud(embeddings["claim"])

    def classify_inv(self, embeddings):
        return self.head_inv(embeddings["claim"])


# ── Instantiate ───────────────────────────────────────────────────────────────
HIDDEN_DIM = 256
OUT_DIM    = 128

model = ResHeteroSAGE(
    hidden_dim=HIDDEN_DIM,
    out_dim=OUT_DIM,
    metadata=data.metadata(),
    dropout=0.3,
).to(device)

data = data.to(device)

# Materialise lazy SAGEConv parameters via one silent forward pass
with torch.no_grad():
    _ = model(data.x_dict, data.edge_index_dict)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
vram_mb  = torch.cuda.memory_allocated() / 1e6 if device.type == "cuda" else 0
print(f"ResHeteroSAGE v2")
print(f"  Hidden: {HIDDEN_DIM}  |  Output: {OUT_DIM}")
print(f"  Layers: 4 (layer2+3 have residual connections)")
print(f"  Parameters: {n_params:,}")
print(f"  VRAM after init: {vram_mb:.0f} MB")

In [ ]:
# Train ResHeteroSAGE v2
# Changes vs v1:
#   - Focal loss (gamma=2) instead of plain cross-entropy
#     -> focuses gradient on hard / misclassified examples
#   - OneCycleLR (warmup 10% + cosine decay) instead of CosineAnnealingLR
#     -> better learning rate schedule for deeper models
#   - 200 epochs vs 50 (more training for deeper model to converge)
#   - Save to gnn_embeddings_v2.parquet (128-dim)
print("=" * 60)
print("Training ResHeteroSAGE v2  (200 epochs, focal loss)")
print("=" * 60)

EPOCHS   = 200
MAX_LR   = 0.005
GAMMA_FL = 2.0       # focal loss exponent
OUT_PATH = r"C:\Users\mitul\OneDrive\Documents\arya\fraud\gnn_embeddings_v2.parquet"


# ── Focal loss ────────────────────────────────────────────────────────────────
def focal_loss(logits, labels, weight=None, gamma=2.0):
    """Focal loss: (1-p_t)^gamma * CE — down-weights easy examples."""
    ce = F.cross_entropy(logits, labels, weight=weight, reduction="none")
    pt = torch.exp(-ce)                       # pt = model confidence on true class
    fl = ((1.0 - pt) ** gamma) * ce
    return fl.mean()


# ── Class weights (capped) ────────────────────────────────────────────────────
def class_weights(y_arr, mask, cap=10.0):
    pos    = y_arr[mask].sum()
    neg    = mask.sum() - pos
    raw_w  = neg / max(pos, 1)
    capped = min(float(raw_w), cap)
    return torch.tensor([1.0, capped], dtype=torch.float).to(device)

cw_fraud = class_weights(y_fraud, train_mask.values)
cw_inv   = class_weights(y_inv,   train_mask.values)
print(f"  Fraud class weight : {cw_fraud[1]:.1f}")
print(f"  Inv   class weight : {cw_inv[1]:.1f}")


# ── Optimiser + scheduler ─────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr      = MAX_LR,
    total_steps = EPOCHS,
    pct_start   = 0.10,        # 10% warmup
    anneal_strategy = "cos",
    div_factor  = MAX_LR / 1e-4,   # start from 1e-4
    final_div_factor = 1e3,         # end at ~5e-6
)


# ── Training loop ─────────────────────────────────────────────────────────────
best_pr_fraud   = 0.0
best_embeddings = None
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    embeddings   = model(data.x_dict, data.edge_index_dict)
    train_idx_t  = data["claim"].train_mask

    # Fraud focal loss
    logits_fraud = model.classify_fraud(embeddings)[train_idx_t]
    labels_fraud = data["claim"].y_fraud[train_idx_t]
    loss_fraud   = focal_loss(logits_fraud, labels_fraud, weight=cw_fraud, gamma=GAMMA_FL)

    # Investigation focal loss (auxiliary — 0.5 weight)
    logits_inv   = model.classify_inv(embeddings)[train_idx_t]
    labels_inv   = data["claim"].y_inv[train_idx_t]
    loss_inv     = focal_loss(logits_inv, labels_inv, weight=cw_inv, gamma=GAMMA_FL)

    loss = loss_fraud + 0.5 * loss_inv
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    # Evaluate every 20 epochs
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            emb         = model(data.x_dict, data.edge_index_dict)
            te          = data["claim"].test_mask
            prob_fraud  = F.softmax(model.classify_fraud(emb)[te], dim=1)[:, 1].cpu().numpy()
            prob_inv    = F.softmax(model.classify_inv(emb)[te],   dim=1)[:, 1].cpu().numpy()
            y_te_fraud  = data["claim"].y_fraud[te].cpu().numpy()
            y_te_inv    = data["claim"].y_inv[te].cpu().numpy()

            pr_fraud = average_precision_score(y_te_fraud, prob_fraud)
            pr_inv   = average_precision_score(y_te_inv,   prob_inv)

            if pr_fraud > best_pr_fraud:
                best_pr_fraud   = pr_fraud
                best_embeddings = emb["claim"].detach().cpu().numpy().copy()

        elapsed = time.time() - t0
        cur_lr  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {loss.item():.4f} | LR: {cur_lr:.5f} | "
              f"PR-AUC Fraud: {pr_fraud:.4f} | Inv: {pr_inv:.4f} | {elapsed:.0f}s")

print(f"\nBest Test PR-AUC (Fraud): {best_pr_fraud:.4f}")
print(f"Total time: {(time.time()-t0)/60:.1f} min")
print("Training complete")


# ── Save v2 embeddings ────────────────────────────────────────────────────────
print("\nSaving gnn_embeddings_v2.parquet ...")
emb_cols = [f"gnn_emb_{i}" for i in range(OUT_DIM)]
emb_df   = pd.DataFrame(best_embeddings, columns=emb_cols)
emb_df.insert(0, "Claim_No", claim_ids)

emb_df.to_parquet(OUT_PATH, index=False)
print(f"  Saved {emb_df.shape[0]:,} rows x {emb_df.shape[1]} cols")
print(f"  Path: {OUT_PATH}")
print(f"  Columns: gnn_emb_0 ... gnn_emb_{OUT_DIM - 1}")